# 第二课：揭开基础模型的面纱 —— 模型是如何炼成的

## 学习目标
- 理解大语言模型的「概率本质」：为什么同一个问题问两次，答案可能不同
- 亲手调节 Temperature、Top-P 等参数，感受对输出的影响
- 触发并识别 AI 的「幻觉」（一本正经地胡说八道）
- 验证「temperature=0 也不保证两次输出完全一致」，理解概率本质
>
> 不同规模模型的对比属于概念轨（见教程文档「体验三：模型大小对比实验」）。

> 所有代码单元都配有详细中文注释，适合零编程基础的学员。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：亲手调节模型的「创造力」——Temperature 实验

### 活动目标
Temperature（温度）是控制 AI 输出创造力的核心参数：
- temperature=0：最保守，每次回答几乎一样
- temperature=1：有创意
- temperature=1.5+：开始「狂野」，可能出现奇怪的内容

让我们用同一个 prompt 在不同温度下测试，直观感受差异。

In [ ]:
prompt = '请写一首关于秋天的四行短诗。'

# 在不同温度下测试
for temp in [0, 0.3, 0.7, 1.0, 1.5]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temp
    )
    print(f'\n温度 = {temp}')
    print('-' * 30)
    print(response.choices[0].message.content)
    print()

### 讨论

- temperature=0 的诗是否最「安全」、最常规？
- temperature=1.5 的诗是否更有「惊喜」（或惊吓）？
- 什么场景应该用低温度？什么场景适合高温度？

> Temperature 就像 AI 的「冒险精神」——温度越低越保守，温度越高越敢胡说。

In [ ]:
# 补充实验：Temperature=0 时，两次回答是否完全一样？
print('测试 Temperature=0 的确定性：')
q = '请用一句话概括机器学习的定义。'

r1 = client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':q}], temperature=0)
r2 = client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':q}], temperature=0)

print(f'第一次：{r1.choices[0].message.content}')
print(f'第二次：{r2.choices[0].message.content}')
if r1.choices[0].message.content == r2.choices[0].message.content:
    print('\n两次回答完全一致！Temperature=0 让模型变得确定性。')
else:
    print('\n两次回答不完全一致（实际场景中 temperature=0 也未必绝对确定）。')

---

## 活动二：触发并识别 AI 的「幻觉」

### 活动目标
AI 有时会「一本正经地胡说八道」——这叫幻觉（Hallucination）。
它不是一个 bug，而是 AI 作为「文本生成器」的自然结果：AI 的目标是生成看起来合理的文本，而不是事实正确的文本。

In [ ]:
# 测试一：问一个不存在的章节
print('【测试一：不存在的书章节】')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'请列出《AI Engineering》这本书第15章的核心内容。这本书实际只有10章。'}],
    temperature=0.3)
print(r.choices[0].message.content)
print('\n真相：这本书只有10章！如果 AI 编造了内容，那就是幻觉。')

# 测试二：不存在的论文
print('\n【测试二：不存在的论文】')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'请评价一篇名为「Neural Quantum Entanglement in LLMs」的2024年论文。这篇论文是我编造的，并不存在。'}],
    temperature=0.3)
print(r.choices[0].message.content)
print('\n真相：这篇论文根本不存在！')

# 测试三：引导 AI 承认不知道
print('\n【测试三：引导 AI 说我不知道】')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'2027年奥运会在哪个城市举办？如果你不确定请直接说不知道。'}],
    temperature=0)
print(r.choices[0].message.content)
print('\n实际上2028年奥运会才在洛杉矶举办，没有2027年奥运会。')

### 讨论

- AI 在哪个场景最容易产生幻觉？为什么？
- 如何让 AI 更诚实地承认「不知道」？
- 把 AI 想象成一个「非常会编故事的朋友」——他说的话听起来很有道理，但你得自己核实事实。

---

## 活动三：体验 Top-P —— 另一个控制创造力的旋钮

### 活动目标
Top-P（nucleus sampling）限制 AI 只从累积概率达到 P 的那些词中选择。
- Top-P=0.1：只从最可能的 10% 词汇中选——非常保守
- Top-P=0.9：从 90% 的词汇中选择——更有创意

In [ ]:
prompt = '用一句话描述人工智能的未来。'

for top_p in [0.1, 0.5, 0.9]:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content':prompt}],
        top_p=top_p,
        temperature=0.7
    )
    print(f'\nTop-P = {top_p}')
    print('-' * 40)
    print(r.choices[0].message.content)

print('\nTop-P 越小 -> 词汇选择越窄 -> 回答越可预测')
print('Top-P 越大 -> 词汇选择越宽 -> 回答越多样')

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| Temperature 调节 | 理解温度如何影响创造力和确定性 |
| Top-P 调节 | 理解核采样如何限制词汇选择范围 |
| 识别幻觉 | 学会触发并辨别 AI 在「编造」内容 |
| 概率本质 | 理解为什么同一个问题两次回答可能不同 |

### 课后练习
1. 尝试 temperature=2.0（最大值），观察输出是否会变得完全不可理解
2. 搜索「ChatGPT hallucinations examples」，看看其他人遇到了什么有趣的幻觉
3. 思考：在你自己的工作中，如果 AI 产生幻觉，可能造成什么后果？